# 01 — Исследование датасета ForteBank

Проверяем видео, разметку, обычные действия и возможные подсказки для модели до обучения.
Исходная разметка — `data/annotations/events.csv`: ноутбук её **не перезаписывает**.
Расчётные таблицы и предложение разбиения сохраняются в `data/processed/`, изображения — в `reports/figures/`.

Видео ищутся в `open/`, `data/open/`, `data/raw/open/`, `data/raw/videos/`, `data/raw/`.
Путь можно задать явно через `VIDEO_DIR`. Некорректный явный путь вызывает ошибку.
Описания событий и категории — предварительные интерпретации выданных описаний; они не подтверждают ручной просмотр видео.
Для обучения используются исходные `source_label` и `timecode`; описания и категории служат только для разведочного анализа.
Описания и нормативные ссылки требуют проверки; исходные метки не меняются автоматически.
`events.csv` принят командой как источник меток и единственных отметок событий. Повторное утверждение в отдельной таблице не требуется. EDA проверяет техническую корректность данных; готовность независимого split оценивается отдельно.


In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src/overfitted_underqualified").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите из корня репозитория или notebooks/")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import cv2
import pandas as pd
import seaborn as sns
from IPython.display import display

from overfitted_underqualified.eda.plots import (
    contact_sheets,
    plot_class_parameters,
    plot_distributions,
    plot_scene_candidates,
    plot_zones,
)
from overfitted_underqualified.eda.reports import (
    inspect_directory,
    prepare_annotations,
    write_reports,
)
from overfitted_underqualified.eda.split import prepare_split, scene_candidates


In [ ]:
RANDOM_STATE = 42

WINDOW_SEC, FRAME_STEP_SEC = 4.0, 0.5

SCENE_DISTANCE_THRESHOLD = 0.12

ANNOTATIONS_DIR = ROOT / "data" / "annotations"

PROCESSED_DIR = ROOT / "data" / "processed"

FIGURES_DIR = ROOT / "reports" / "figures"

EVENT_FRAMES_DIR = FIGURES_DIR / "event_frames"

NORMAL_FRAMES_DIR = FIGURES_DIR / "normal_frames"

EVENTS_PATH = ANNOTATIONS_DIR / "events.csv"


ZONES_PATH = ANNOTATIONS_DIR / "zones.json"

for folder in (PROCESSED_DIR, FIGURES_DIR, EVENT_FRAMES_DIR, NORMAL_FRAMES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

pd.set_option("display.max_colwidth", 120)

pd.set_option("display.max_rows", 60)

print(f"OpenCV {cv2.__version__}; результаты: {PROCESSED_DIR}")


## 1. Техническая проверка и точные копии

Для каждого файла считаем SHA-256, проверяем метаданные и декодируем пять кадров от начала до 98% длительности.
`readable` означает, что метаданные корректны и прочитались все пять проб. Это выборочная проверка,
а не гарантия целостности каждого кадра. Последний расчётный кадр проверяется отдельно. При неудачном переходе по таймкоду
используется последовательное декодирование и фактические временные метки кадров.
Предупреждение остаётся только если резервное чтение тоже не помогло.
Длительность считается как число кадров / FPS;
для переменной частоты кадров она может быть приблизительной.

In [ ]:
DATA_DIR, inventory, readable, thumbnails = inspect_directory(ROOT, PROCESSED_DIR)


## 2. Разметка и отчёт об ошибках

Таймкод должен быть корректным и попадать внутрь ролика. Для нарушения он обязателен.
Повторяющиеся ID, конфликтующие метки точных копий, отсутствующие файлы и ошибки чтения
попадают в `validation_issues.csv`. Проблемные ролики исключаются из дальнейшего анализа и разбиения;
ошибки остаются видны в отчёте и итоговом чеклисте.

In [ ]:
events, analysis, training_videos, issues, source_digest = prepare_annotations(EVENTS_PATH, inventory, PROCESSED_DIR)


### Исследовательские вопросы к разметке
Сравним исходные строки и уникальные видео: точные копии не должны увеличивать вес класса.
Категории описаний используются для анализа разнообразия, не как готовые multiclass targets.
Меняйте выбранный ID ниже, чтобы сопоставлять текст события и визуальный контекст локально.

In [ ]:
label_balance = pd.concat({
    "source_rows": events["source_label"].value_counts(),
    "unique_videos": training_videos["source_label"].value_counts(),
}, axis=1).fillna(0).astype(int)
display(label_balance)
annotation_coverage = events.groupby("source_label").agg(
    rows=("clip_id", "size"), timestamps=("timecode_sec", "count"),
    event_types=("observed_event_type", "nunique"),
)
display(annotation_coverage)
display(pd.crosstab(events["observed_event_type"], events["source_label"]))
positive_positions = analysis.loc[analysis["source_label"].eq("violation"),
    ["clip_id", "timecode_sec", "duration_sec"]].copy()
positive_positions["relative_position"] = positive_positions["timecode_sec"] / positive_positions["duration_sec"]
display(positive_positions.sort_values("relative_position"))


## 3. Длительность, FPS, разрешение и баланс меток

Графики строятся заново при каждом запуске. Проверяем оба класса, редкие типы событий
и относительное положение события внутри ролика. Данные исключённых файлов остаются в отчёте ошибок.

In [ ]:
generated_figures = plot_distributions(readable, events, analysis, FIGURES_DIR)


## 4. Визуальный просмотр обоих классов

Для нарушения берём окно ±4 секунды с шагом 0,5 секунды, обязательно включая сам таймкод.
Кадры не выходят за последнюю позицию видео. Для обычного ролика берём пять кадров по всей длительности.
Это обзор: короткое действие всё ещё может требовать просмотра исходного видео с полной частотой кадров.
Показываются по два контактных листа каждого класса, остальные сохраняются в папки `event_frames/` и `normal_frames/`.
`frame_extraction.csv` фиксирует каждую попытку, число успешных кадров и полноту последовательности.

Обычные действия тоже нужно описывать: передача денег и уход от стола могут встречаться в обоих классах.

In [ ]:
frame_report = contact_sheets(analysis, DATA_DIR, EVENT_FRAMES_DIR, NORMAL_FRAMES_DIR, PROCESSED_DIR, WINDOW_SEC, FRAME_STEP_SEC)
display(frame_report)


In [ ]:
from overfitted_underqualified.eda.plots import save_contact_sheet, window_timestamps

SELECTED_CLIP = "X-4"  # Измените ID для локального просмотра.
selected_event = analysis.loc[analysis["clip_id"].eq(SELECTED_CLIP)].iloc[0]
display(events.loc[events["clip_id"].eq(SELECTED_CLIP)])
center = selected_event["timecode_sec"]
if pd.isna(center):
    center = selected_event["duration_sec"] / 2
stamps = window_timestamps(center, selected_event["duration_sec"], selected_event["fps"],
                          window=WINDOW_SEC, step=FRAME_STEP_SEC)
preview_dir = FIGURES_DIR / "research_previews"
preview_dir.mkdir(exist_ok=True)
display(save_contact_sheet(SELECTED_CLIP, DATA_DIR / selected_event["video_file"],
                           stamps, preview_dir, show=True, root=ROOT))


## 5. Возможные подсказки для модели

Сравниваем технические параметры, яркость и резкость между классами по уникальным видео, без повторного учёта точных копий. Эти показатели не доказывают
причинную связь, но помогают заметить ситуацию, когда класс угадывается по длине ролика или освещению.
Резкость зависит и от содержания кадра; универсальный порог «плохого видео» здесь не задаём.
Фон и обстановку проверяем также по парам похожих сцен в следующем разделе.

In [ ]:
plot_class_parameters(analysis, PROCESSED_DIR, FIGURES_DIR, generated_figures)


## 6. Похожие сцены и предложение разделения

Точные копии объединяются по SHA-256. В `training_videos.csv` и разбиении остаётся
одна строка на уникальное видео; `source_clip_ids` и `event_times_sec` сохраняют все
исходные ID и отметки событий. Все исходные записи остаются в `events.csv`,
а соответствие ID хранится в `duplicate_map.csv`. Обработанные копии отражаются
в `dataset_notes.csv` как информация, а конфликт меток остаётся ошибкой. Для поиска похожей обстановки сравниваем уменьшенные
медианные изображения пяти кадров, без использования меток. Это эвристика: возможны лишние пары
и пропуски. `scene_candidates.csv` и обзор кадров помогают вручную проверить фон, камеру и помещение.

Согласован предварительный протокол `within_room_unique_video`: train/validation примерно 75/25 по уникальным SHA-256, seed=42. Точные копии всегда вместе; обе выборки содержат оба класса. Визуальное сходство сохраняется только как диагностика и не объединяет ролики для этого split. Номера сеансов неизвестны: связанные записи, люди и общий фон могут пересекаться. Метрики характеризуют только проверку внутри этого кабинета, не независимые сеансы или новые камеры.


In [ ]:
usable_ids = set(analysis["clip_id"])
pairs = scene_candidates({k: v for k, v in thumbnails.items() if k in usable_ids}, SCENE_DISTANCE_THRESHOLD)
pairs.to_csv(PROCESSED_DIR / "scene_candidates.csv", index=False)
display(pairs)
plot_scene_candidates(pairs, thumbnails, FIGURES_DIR)
groups, split, split_status = prepare_split(analysis, inventory, training_videos, PROCESSED_DIR, RANDOM_STATE)
display(pd.crosstab(split["split"], split["source_label"]))
print(split_status)


## 7. Зоны кадра из отдельного файла

Зоны при необходимости задаются в `data/annotations/zones.json` как
`{"clip_id": {"zone_name": [x1, y1, x2, y2]}}`. Координаты проверяются по размеру кадра.
Без файла этот дополнительный этап пропускается; координаты не вшиты в ноутбук.

In [ ]:
plot_zones(ZONES_PATH, analysis, DATA_DIR, FIGURES_DIR)


## Итоговые проверки

Отчёт фиксирует seed, хеши, полноту чтения и готовность к предварительному within-room baseline. Независимая оценка не подтверждена.


In [ ]:
readiness, checklist = write_reports(
    events, readable, frame_report, EVENTS_PATH, source_digest, issues, split,
    training_videos, split_status, RANDOM_STATE, groups, PROCESSED_DIR, inventory,
    generated_figures, DATA_DIR, SCENE_DISTANCE_THRESHOLD,
)
assert checklist["Готово"].all(), "Исправьте проваленные проверки перед обучением"


## Далее: обучение и оценка

Согласованный предварительный split содержит 30 train и 10 validation уникальных видео.
`events.csv` остаётся единственным источником меток и отметок. Разные события X-4 и X-47 сохранены.
Перейдите к `02_detection.ipynb`, затем `03_evaluation.ipynb`.

Оценивается классификация окон вокруг **известных отметок**, а не поиск событий во всём видео.
Кабинет, люди и неизвестные сеансы могут пересекаться между выборками. Независимость не подтверждена.
Первый smoke-run предсказывал нарушение для всех validation-окон: это проверка работоспособности,
а не хорошее качество и не законченный детектор.


### Выводы EDA и рабочие заметки
В текущем наборе 42 исходные строки соответствуют 40 уникальным видео. Для обучения важны
бинарные метки и все разные отметки канонического ролика; редкие описательные категории
не обосновывают обучение многоклассовой модели. SHA-256 защищает только от точных копий.
После просмотра выбранного ролика запишите здесь наблюдение, сомнение и следующую проверку.
Не меняйте метки автоматически по предсказаниям модели. Таблицы и банковские кадры остаются
локальными: перед коммитом очистите все outputs, execution counts и attachments.